# Out-of-sample performance of industry portfolio strategies

We compare eight ways to allocate wealth across ten US industry portfolios. Each month, we estimate portfolio weights using the preceding 240 months of data and hold those weights for one month. All portfolios invest 100% of wealth in the risky assets. Some strategies allow short positions.

The question is whether estimating expected returns, covariances, or factor exposures improves the investor's realized certainty equivalent relative to equal weighting. We evaluate performance both before and after transaction costs of 50 basis points per dollar traded.


In [3]:
from pathlib import Path
from io import BytesIO, StringIO
from urllib.request import urlopen
from zipfile import ZipFile
import re

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from IPython.display import display, Markdown

START = "1965-01"
END = "2025-01"
WINDOW = 240
COST = 0.005

INDUSTRY_FILE = Path("C:/Users/gioff/Desktop/CODING/PM/PM_case/data/10_Industry_Portfolios.csv")

## 1. Data and return definitions

We use monthly industry total returns, the US Fama–French three factors, the US momentum factor, and the one-month risk-free rate. All calculations use decimal returns.

Let $R_{i,t}$ denote industry $i$'s total return and $r_{f,t}$ the risk-free rate. The industry excess return is

$$
x_{i,t}=R_{i,t}-r_{f,t}.
$$

We use excess returns to estimate the portfolios and evaluate their performance. We retain total returns to calculate how the value of each holding changes between rebalances. The market factor `Mkt-RF` is already an excess return.

In [4]:
industry_pct = pd.read_csv(INDUSTRY_FILE, index_col="yyyymm")
industry_pct.index = pd.to_datetime(
    industry_pct.index.astype(str), format="%Y%m"
).to_period("M")
industry_pct.columns = industry_pct.columns.str.strip()

FRENCH_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"


def read_french_monthly(dataset, columns):
    """Read the monthly rows of one official Kenneth French CSV archive."""
    archive = Path(f"{dataset}_CSV.zip")

    if archive.exists():
        raw_zip = archive.read_bytes()
    else:
        with urlopen(FRENCH_URL + archive.name, timeout=60) as response:
            raw_zip = response.read()

    with ZipFile(BytesIO(raw_zip)) as z:
        csv_name = next(name for name in z.namelist() if name.lower().endswith(".csv"))
        text = z.read(csv_name).decode("utf-8-sig", errors="replace")

    # Monthly rows begin with YYYYMM. Annual rows begin with YYYY.
    monthly_rows = [
        line for line in text.splitlines()
        if re.match(r"^\s*\d{6}\s*,", line)
    ]
    data = pd.read_csv(StringIO("\n".join(monthly_rows)), header=None)
    data.columns = ["date", *columns]
    data.index = pd.to_datetime(
        data.pop("date").astype(str), format="%Y%m"
    ).dt.to_period("M")

    if not archive.exists():
        archive.write_bytes(raw_zip)

    return data


ff3_pct = read_french_monthly(
    "F-F_Research_Data_Factors", ["Mkt-RF", "SMB", "HML", "RF"]
)
mom_pct = read_french_monthly("F-F_Momentum_Factor", ["Mom"])

months = pd.period_range(START, END, freq="M")
industries = industry_pct.columns.tolist()
factor_names = ["Mkt-RF", "SMB", "HML", "Mom"]

# Reindex to the required calendar so a missing month becomes visible.
for source in [industry_pct, ff3_pct, mom_pct]:
    assert source.index.is_unique, "Duplicate month in an input file."

data_pct = pd.concat([industry_pct, ff3_pct, mom_pct], axis=1).reindex(months)
data_pct = data_pct.replace([-99.99, -999, -999.99], np.nan)
assert data_pct.notna().all().all(), "Missing return or factor in the sample."
assert len(industries) == 10 and len(data_pct) == 721

data = data_pct / 100
total_returns = data[industries]
risk_free = data["RF"]
excess_returns = total_returns.sub(risk_free, axis="index")
factors = data[factor_names]

print(f"Sample: {months[0]} to {months[-1]} ({len(months)} months)")
print(f"First estimation window: {months[0]} to {months[WINDOW - 1]}")
print(f"Holding period: {months[WINDOW]} to {months[-1]}")
print(f"Out-of-sample months: {len(months) - WINDOW}")

Sample: 1965-01 to 2025-01 (721 months)
First estimation window: 1965-01 to 1984-12
Holding period: 1985-01 to 2025-01
Out-of-sample months: 481


## 2. Portfolio weights

At the beginning of each holding month, let $\hat\mu$ and $\hat\Sigma$ denote the sample mean and covariance of monthly industry excess returns over the previous 240 months. Let $\mathbf 1$ be a vector of ones and define

$$
a=\hat\Sigma^{-1}\mathbf 1,\qquad
b=\hat\Sigma^{-1}\hat\mu,\qquad
A=\mathbf 1'a,\qquad B=\mathbf 1'b.
$$

### (a) Equal weight, global minimum variance, and tangency

$$
w_{ew}=\frac{\mathbf 1}{N},\qquad
w_{gmv}=\frac{a}{A},\qquad
w_{tan}=\frac{b}{B}.
$$

Equal weighting estimates no parameters. GMV uses the covariance matrix. The normalized tangency rule also uses expected excess returns. When $B$ is close to zero, normalizing the tangency weights to sum to one can produce large offsetting long and short positions.

### (b) Long-only GMV and certainty-equivalent portfolios

The long-only GMV portfolio solves

$$
\min_w\; w'\hat\Sigma w
\quad\text{subject to}\quad
\mathbf 1'w=1,\quad w_i\geq 0.
$$

For risk aversion $\gamma$, the fully invested CE portfolio solves

$$
\max_w\; w'\hat\mu-\frac{\gamma}{2}w'\hat\Sigma w
\quad\text{subject to}\quad \mathbf 1'w=1.
$$

The first-order condition gives
$w=(b-\lambda a)/\gamma$. Imposing the budget constraint gives
$\lambda=(B-\gamma)/A$. Substitution yields

$$
w_{ce(\gamma)}
=w_{gmv}+\frac{1}{\gamma}\left(b-Bw_{gmv}\right).
$$

The term in parentheses sums to zero. It shifts wealth across industries while preserving full investment. Doubling risk aversion from 4 to 8 halves this tilt away from GMV.

In [6]:
def long_only_gmv(cov):
    """Minimum-variance weights subject to full investment and no short sales."""
    n = len(cov)
    equal_weights = np.full(n, 1 / n)
    scaled_cov = cov / np.diag(cov).mean()

    result = minimize(
        fun=lambda w: w @ scaled_cov @ w,
        x0=equal_weights,
        jac=lambda w: 2 * scaled_cov @ w,
        method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={
            "type": "eq",
            "fun": lambda w: w.sum() - 1,
            "jac": lambda w: np.ones(n),
        },
        options={"ftol": 1e-12, "maxiter": 500},
    )

    if not result.success:
        raise RuntimeError(f"Long-only GMV failed: {result.message}")

    return result.x


def markowitz_weights(mu, cov):
    """Compute the six portfolio rules in parts (a) and (b)."""
    n = len(mu)
    ones = np.ones(n)

    # solve(S, v) gives S^{-1}v without explicitly forming an inverse.
    a = np.linalg.solve(cov, ones)
    b = np.linalg.solve(cov, mu)

    gmv = a / a.sum()
    B = b.sum()
    if np.isclose(B, 0, atol=1e-12, rtol=0):
        raise ValueError("Tangency normalization is numerically undefined.")

    tilt = b - B * gmv

    return {
        "EW": ones / n,
        "GMV": gmv,
        "GMV+": long_only_gmv(cov),
        "TAN": b / B,
        "CE(4)": gmv + tilt / 4,
        "CE(8)": gmv + tilt / 8,
    }

### (c) Reward-to-risk timing based on average factor exposure

Within each estimation window, we regress each industry's excess return on the same four factors:

$$
x_{i,s}=\alpha_i+\beta_{i,M}(Mkt-RF)_s+\beta_{i,S}SMB_s
+\beta_{i,H}HML_s+\beta_{i,U}Mom_s+\varepsilon_{i,s}.
$$

For each industry, $\bar\beta_i$ is the arithmetic average of the four estimated slopes. We take its positive part and divide by the industry's estimated excess-return variance, $\hat\sigma_i^2=\hat\Sigma_{ii}$. With timing parameter $\vartheta=2$,

$$
q_i=\left(\frac{\max(\bar\beta_i,0)}{\hat\sigma_i^2}\right)^2,
\qquad
w_{rrt,i}=\frac{q_i}{\sum_j q_j}.
$$

Taking the positive part before squaring prevents a negative average exposure from generating a positive reward signal. The variance in the denominator measures the industry's total excess-return risk, including both factor and residual risk.

### (d) Own strategy: inverse-variance weighting

We choose a long-only portfolio that gives more weight to industries with lower estimated variance:

$$
w_{own,i}=
\frac{1/\hat\sigma_i^2}{\sum_j 1/\hat\sigma_j^2}.
$$

This is the GMV solution when the covariance matrix is approximated by its diagonal. It provides a direct comparison with unrestricted GMV. With ten industries, the rule estimates ten variances and omits 45 pairwise covariances. This can reduce sensitivity to estimation error, at the cost of ignoring diversification opportunities between industries. The rule is fixed before inspecting out-of-sample performance.

In [7]:
def timing_weights(window_excess, window_factors, variances):
    """Compute the average-beta RRT portfolio and the inverse-variance portfolio."""
    # One intercept plus four factor columns.
    X = np.column_stack([np.ones(len(window_factors)), window_factors])

    # Each column of Y is one industry. This fits ten separate OLS regressions.
    Y = window_excess
    coefficients, _, rank, _ = np.linalg.lstsq(X, Y, rcond=None)
    if rank != X.shape[1]:
        raise ValueError("Factor regression has a rank-deficient design matrix.")

    slopes = coefficients[1:, :]          # 4 factors × 10 industries
    average_beta = slopes.mean(axis=0)    # one average per industry

    positive_beta = np.maximum(average_beta, 0)
    rrt_score = (positive_beta / variances) ** 2
    if rrt_score.sum() == 0:
        raise ValueError("All RRT scores are zero in this window.")

    inverse_variance = 1 / variances

    return {
        "RRT": rrt_score / rrt_score.sum(),
        "OWN": inverse_variance / inverse_variance.sum(),
    }

## 3. Rolling out-of-sample experiment

For holding month $t$, we estimate all inputs from months $t-240,\ldots,t-1$. We then apply the resulting weights $w_t$ to the realized industry excess returns in month $t$:

$$
x_{p,t}=w_t'x_t.
$$

For example, the January 1985 weights use only January 1965 through December 1984. January 1985 enters the estimation sample for the first time when we choose the February 1985 weights.

We save every target weight. This allows us to reconstruct realized returns, measure rebalancing trades, and check the optimization results independently.

In [8]:
strategy_names = ["EW", "GMV", "GMV+", "TAN", "CE(4)", "CE(8)", "RRT", "OWN"]
holding_months = months[WINDOW:]

weights = {
    name: pd.DataFrame(index=holding_months, columns=industries, dtype=float)
    for name in strategy_names
}

for t in range(WINDOW, len(months)):
    holding_month = months[t]

    # The right endpoint t is excluded from both training slices.
    window_excess = excess_returns.iloc[t - WINDOW:t]
    window_factors = factors.iloc[t - WINDOW:t]

    mu = window_excess.mean().to_numpy()
    cov = window_excess.cov().to_numpy()

    current = markowitz_weights(mu, cov)
    current.update(
        timing_weights(
            window_excess.to_numpy(),
            window_factors.to_numpy(),
            np.diag(cov),
        )
    )

    # In sample, the unrestricted GMV has the largest feasible set.
    # Equal weights are feasible for the long-only GMV problem.
    var_gmv = current["GMV"] @ cov @ current["GMV"]
    var_long = current["GMV+"] @ cov @ current["GMV+"]
    var_ew = current["EW"] @ cov @ current["EW"]

    assert var_gmv <= var_long + 1e-10, holding_month
    assert var_long <= var_ew + 1e-10, holding_month

    for name, w in current.items():
        assert np.isfinite(w).all(), (holding_month, name)
        assert np.isclose(w.sum(), 1, atol=1e-8, rtol=0), (holding_month, name)
        weights[name].loc[holding_month] = w

for name in ["GMV+", "RRT", "OWN"]:
    assert weights[name].to_numpy().min() >= -1e-8, name

np.testing.assert_allclose(
    weights["CE(8)"],
    0.5 * (weights["CE(4)"] + weights["GMV"]),
    atol=1e-10,
)

print(f"Verified {len(holding_months)} monthly allocations for each strategy.")

Verified 481 monthly allocations for each strategy.


## 4. Turnover, transaction costs, and certainty equivalents

An industry's portfolio weight changes during a month as its return differs from the portfolio return. If $w_{i,t}$ is its starting weight, its weight at month-end is

$$
w^{drift}_{i,t}
=\frac{w_{i,t}(1+R_{i,t})}{1+w_t'R_t}.
$$

The turnover required to establish the next target is

$$
TO_{t+1}=\sum_i |w_{i,t+1}-w^{drift}_{i,t}|.
$$

We count both purchases and sales, as in the turnover definition in Lecture 6. With a proportional transaction cost of $c=0.005$, the net excess return for holding month $t$ is

$$
x^{net}_{p,t}=x_{p,t}-c\,TO_t.
$$

We treat the first portfolio as already established. January 1985 therefore incurs no opening cost. Average turnover uses the 480 observable rebalances from February 1985 to January 2025. Gross and net CE both use the same 481 holding months.

For a series of monthly excess returns $z_t$, the annualized certainty equivalent in percentage points is

$$
CE^{ann}(\gamma)
=100\cdot12\left(\bar z-\frac{\gamma}{2}s_z^2\right),
$$

where $s_z^2$ is the sample variance. We apply the formula separately to gross and net returns. Trading costs can change both the mean and the variance.

In [9]:
oos_total = total_returns.loc[holding_months]
oos_excess = excess_returns.loc[holding_months]

gross_excess = pd.DataFrame(index=holding_months, columns=strategy_names, dtype=float)
turnover = gross_excess.copy()

for name, w in weights.items():
    gross_excess[name] = (w * oos_excess).sum(axis=1)
    portfolio_total = (w * oos_total).sum(axis=1)

    # A nonpositive gross wealth factor would invalidate ordinary weight drift.
    if (1 + portfolio_total <= 0).any():
        failed_months = portfolio_total.index[1 + portfolio_total <= 0].tolist()
        raise ValueError(f"{name}: nonpositive portfolio wealth in {failed_months}.")

    drifted = (w * (1 + oos_total)).div(1 + portfolio_total, axis="index")

    # At the start of t, compare target t to drifted holdings from t-1.
    # min_count preserves the missing first rebalance.
    turnover[name] = (w - drifted.shift(1)).abs().sum(
        axis=1, min_count=len(industries)
    )

assert turnover.iloc[0].isna().all()
assert turnover.iloc[1:].notna().all().all()

trading_cost = COST * turnover.copy()
trading_cost.iloc[0] = 0.0
net_excess = gross_excess - trading_cost

# Independent benchmarks calculated from the uploaded industry CSV.
january_1985_ew_total = oos_total.iloc[0].mean()
np.testing.assert_allclose(january_1985_ew_total, 0.08176, atol=1e-10)
np.testing.assert_allclose(turnover["EW"].iloc[1], 0.02415323177, atol=1e-9)
np.testing.assert_allclose(turnover["EW"].mean(), 0.02459239894, atol=1e-9)

print(f"January 1985 EW total return: {january_1985_ew_total:.2%}")
print(f"February 1985 EW turnover:    {turnover['EW'].iloc[1]:.2%}")
print(f"Average monthly EW turnover:  {turnover['EW'].mean():.2%}")

January 1985 EW total return: 8.18%
February 1985 EW turnover:    2.42%
Average monthly EW turnover:  2.46%


## 5. Table 1. Portfolio performance

Each row represents one portfolio strategy. The two gross CE columns evaluate the same realized returns at two levels of investor risk aversion. The same holds for the two net CE columns. In particular, the portfolio constructed using $\gamma=4$ is evaluated at both $\gamma=4$ and $\gamma=8$.

CE values are annual percentages. Turnover is the average percentage of portfolio value traded per month. All reported figures are rounded to two decimals after calculation.

In [11]:
def annual_ce(returns, gamma):
    monthly_mean = returns.mean()
    monthly_variance = returns.var(ddof=1)
    return 100 * 12 * (monthly_mean - 0.5 * gamma * monthly_variance)


table1 = pd.DataFrame({
    "Gross CE, γ=4 (%)": annual_ce(gross_excess, 4),
    "Gross CE, γ=8 (%)": annual_ce(gross_excess, 8),
    "Monthly turnover (%)": 100 * turnover.mean(),
    "Net CE, γ=4 (%)": annual_ce(net_excess, 4),
    "Net CE, γ=8 (%)": annual_ce(net_excess, 8),
}).loc[strategy_names]

table1.index.name = "Portfolio"

# Raising evaluation risk aversion from 4 to 8 has a known exact effect.
np.testing.assert_allclose(
    table1["Gross CE, γ=4 (%)"] - table1["Gross CE, γ=8 (%)"],
    2400 * gross_excess.var(ddof=1),
)
np.testing.assert_allclose(
    table1["Net CE, γ=4 (%)"] - table1["Net CE, γ=8 (%)"],
    2400 * net_excess.var(ddof=1),
)

display(table1.style.format("{:.2f}"))

,"Gross CE, γ=4 (%)","Gross CE, γ=8 (%)",Monthly turnover (%),"Net CE, γ=4 (%)","Net CE, γ=8 (%)"
Portfolio,,,,,
EW,5.13,0.71,2.46,4.98,0.56
GMV,5.90,2.76,9.02,5.36,2.22
GMV+,5.57,2.44,3.27,5.37,2.24
TAN,1.11,-7.05,50.18,-1.85,-9.97
CE(4),-2.25,-12.27,62.53,-6.00,-16.03
CE(8),3.54,-1.32,29.10,1.80,-3.06
RRT,4.99,0.87,2.89,4.81,0.70
OWN,5.39,1.58,2.34,5.25,1.44


In [12]:
net4 = table1["Net CE, γ=4 (%)"]
net8 = table1["Net CE, γ=8 (%)"]
monthly_to = table1["Monthly turnover (%)"]

best4 = net4.idxmax()
best8 = net8.idxmax()

gmv_ce_change = net8["GMV+"] - net8["GMV"]
gmv_to_change = monthly_to["GMV+"] - monthly_to["GMV"]
own_ce_gap = net8["OWN"] - net8["EW"]

def direction(value, positive, negative):
    if np.isclose(value, 0, atol=1e-10):
        return "was unchanged"
    return f"{positive if value > 0 else negative} by {abs(value):.2f} percentage points"


summary = (
    f"At risk aversion 4, {best4} achieved the highest annual net CE "
    f"({net4[best4]:.2f}%, versus {net4['EW']:.2f}% for equal weighting). "
    f"At risk aversion 8, {best8} ranked first at {net8[best8]:.2f}%. "
    f"After imposing the long-only constraint on GMV, monthly turnover "
    f"{direction(gmv_to_change, 'rose', 'fell')} and annual net CE at risk aversion 8 "
    f"{direction(gmv_ce_change, 'rose', 'fell')}. "
    f"Relative to equal weighting, annual net CE for inverse-variance weighting "
    f"{direction(own_ce_gap, 'was higher', 'was lower')} at risk aversion 8."
)

display(Markdown("### (e) Key findings\n\n" + summary))
print("Summary word count:", len(summary.split()))

tan_exposure = weights["TAN"].abs().sum(axis=1)
print(
    f"Tangency diagnostic: maximum gross exposure was "
    f"{tan_exposure.max():.2f} times wealth in {tan_exposure.idxmax()}."
)

### (e) Key findings

At risk aversion 4, GMV+ achieved the highest annual net CE (5.37%, versus 4.98% for equal weighting). At risk aversion 8, GMV+ ranked first at 2.24%. After imposing the long-only constraint on GMV, monthly turnover fell by 5.75 percentage points and annual net CE at risk aversion 8 rose by 0.02 percentage points. Relative to equal weighting, annual net CE for inverse-variance weighting was higher by 0.89 percentage points at risk aversion 8.

Summary word count: 73
Tangency diagnostic: maximum gross exposure was 17.57 times wealth in 1985-02.
